In [1]:
from transformers import pipeline
import time
import pandas as pd

C:\Users\Levi\miniconda3\envs\text-media\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Hugging Face - Mini-Project
## Help Desk NLP Assistant

In this mini-project, you will build a simple help-desk assistant using **pre-trained Hugging Face pipelines**.

### What you will receive
- **One FAQ context** (a short policy/help document)

### What you must create
- **3 customer complaints** (user messages/tickets)
- **6 user questions** related to the FAQ:
  - **4 questions MUST be answerable** using the FAQ
  - **2 questions MUST be NOT answerable** using the FAQ (to test uncertainty)

## Your Tasks

### 1) Sentiment Detection
For each complaint:
- run sentiment analysis
- report predicted label + score

### 2) Complaint Summarization
For each complaint:
- Summarize in ≤ 60 words and clearly state the main issue.

### 3) Question Answering
For each question:
- run QA using **(question + FAQ context)**
- report answer + score
- **If the question is not answerable** OR the score is low, output:
  - **"Uncertain - not found in the FAQ (needs human review)"**

## Deliverables (submit in your notebook)

1) Your **3 complaints** (the text you created)  
2) Your **6 questions** (clearly marked: answerable vs not answerable)  
3) For each complaint:
   - sentiment output (label + score)
   - summary
4) For each question:
   - QA output (answer + score) or **Uncertain** message
5) Reflection (5–8 lines):
   - What worked well?
   - What failed or was confusing?
   - One improvement you would make (e.g., better questions, clearer FAQ, threshold choice)


In [27]:
# Define three complaints and six questions: four answerable and two unanswerable from the FAQ

# 1) Load FAQ context from a text file
with open("FAQ.txt", "r", encoding="utf-8") as f:
    faq_context = f.read().strip()
print("FAQ loaded. Characters:", len(faq_context))


# 2) Define three customer complaints
complaints = [
    "I placed an order about a week ago and received a shipping confirmation email shortly afterward, which was reassuring at first. However, the tracking information has only updated once and hasn’t provided much clarity about where the package currently is or when it might arrive. I contacted customer support to ask for an estimated delivery date and am still waiting for a response. I’m not particularly upset, but clearer communication or more detailed tracking updates would make the experience feel smoother and more reliable.",
    "When my order arrived, I immediately noticed that the shipping box was damaged, and unfortunately the item inside was broken as well. It appeared that the packaging did not provide enough protection during transit. I contacted customer support and sent photos along with a description of the issue, but I haven’t heard back yet. This situation has been frustrating because I was looking forward to using the product and now feel uncertain about how long it will take to resolve the problem.",
    "I attempted to cancel my order shortly after placing it because I realized I had selected the wrong item. The website mentioned a cancellation window, so I contacted customer support as quickly as possible. Despite this, I was told that the order was already processing and could not be canceled. Now the item has shipped, and I will need to go through the return process, which feels inconvenient and disappointing given how quickly I reached out."
]

# 3) Define six questions related to the FAQ

questions_answerable = [
    "How long does standard shipping usually take?",
    "Within how many days can an item be returned after delivery?",
    "How long does refund processing typically take after a return is received?",
    "Within how much time can an order be canceled after it is placed?",
]

questions_not_answerable = [
    "Do you offer international shipping?",
    "What payment methods are accepted?",
]

# 4) Set the QA confidence threshold
# Return "Uncertain — needs human review" when the score is below this threshold
QA_THRESHOLD = 0.20

# 5) Analyze sentiment and summarize each complaint
sentiment_model = "finiteautomata/beto-sentiment-analysis"
sentiment_pipeline = pipeline("sentiment-analysis", model = sentiment_model)
print("\n-----Sentiment-----")
for c in complaints:
    result = sentiment_pipeline(c)
    print("Label: ", result[0]["label"], "\nConfidence: ", f"{result[0]['score']:.2f}")

summ_model = "google/pegasus-xsum"
summ_pipe = pipeline("summarization", model  = summ_model)

summaries = []

print("\n----------Summary----------")
for c in complaints:
    result = summ_pipe(c, max_length=50, min_length=20, do_sample=False)
    summaries.append(result[0]["summary_text"])
    print("Summary:", result[0]["summary_text"])

print("\n---Summary Sentiment---")
for s in summaries:
    result = sentiment_pipeline(s)
    print("Label: ", result[0]["label"], "\nConfidence: ", f"{result[0]['score']:.2f}")

# 6) Answer each question using only the FAQ as context
qa_model = "distilbert/distilbert-base-cased-distilled-squad"
qa_pipe = pipeline("question-answering", model = qa_model)

print("\n----------Q&A----------")
for q in questions_answerable + questions_not_answerable:
    result = qa_pipe(question=q, context=faq_context)
    if result["score"] > QA_THRESHOLD:
        print("Answer: ", result["answer"])
    else:
        print("Uncertain — needs human review")

# 7) Document results and potential improvements in Markdown

FAQ loaded. Characters: 1575


Device set to use cpu



-----Sentiment-----
Label:  NEU 
Confidence:  0.99
Label:  NEG 
Confidence:  0.89
Label:  NEG 
Confidence:  0.83

-----Summary-----


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


Summary: I’m having a hard time finding out when my order is going to arrive and how long it will take to get there.
Summary: I recently ordered a pair of headphones from Amazon and was very pleased with the quality of the product.
Summary: I am writing to express my dissatisfaction with the service I received from Amazon Canada after placing an order for an item I did not want.

-----Summary Sentiment-----
Label:  NEU 
Confidence:  0.61
Label:  POS 
Confidence:  0.96
Label:  NEG 
Confidence:  0.87


Device set to use cpu



-----Q&A-----
Answer:  3–5 business days
Answer:  30
Answer:  5–7 business days
Answer:  1 hour
Uncertain — needs human review
Uncertain — needs human review


**Reflection**
- *What worked well?*
Sentament analysis worked well and gave reasonable lables, Q&A also produced correct responses and correct "Uncertain" errors.
  
- *What failed or was confusing?*
The summarization model I used began hallucinating that the reviews were specifically for Amazon. That is very likely from training bias.
The second summary not only brought up Amazon, but also a product that wasn't mentioned and changed the sentiment to a positive review.
Because of this I also ran the summaries through the sentiment pipeline to see how they changed.

  
- *One improvement you would make (e.g., better questions, threshold tuning, clearer FAQ)*
I would spend more time finding a better mode for summarization, the two I tried resutled in the first quoting the original text and cutting off
abruptly at the threshold and the other seemed to do better at summarizing until it brought up a company and product that was not mentioned.